# London Experiment: 2-TX Organic Zone Shapes

**Scene:** London — two transmitters with opposing coverage targets  
**Frequency:** 3.5 GHz  

Introduces **organic (polygon) zone shapes** to reflect realistic deployments:
- TX1 covers a **street corner L-shape** (two intersecting street arms)
- TX2 covers an **irregular public square** (pentagon)

From this point on, only gradient baselines are compared and metrics are restricted to
**RSRP** and **SINR** (no zeroth-order methods — those were eliminated in the Duke experiment).

> **Before running:** Set `best_n` below to the sample count selected from the Duke ablation.

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
SCENE_XML_PATH = "../scene/scenes/London/scene.xml"
CARRIER_HZ     = 3.5e9

# Sample count from Duke ablation — set this before running
best_n = 256    # ← set from duke_experiment.ipynb

# Transmitters
TX1_NAME        = "gnb1"
TX1_BUILDING_ID = 10      # ← adjust to a valid building in London scene
TX1_HEIGHT_M    = 10.0

TX2_NAME        = "gnb2"
TX2_BUILDING_ID = 25      # ← adjust to a valid building in London scene
TX2_HEIGHT_M    = 10.0

# ── ZONE CONFIGURATION ──────────────────────────────────────────────────────
# Coordinates are scene-local meters. Run the visualization cell to verify.
#
# Zone 1 — L-shape (street corner: horizontal arm + vertical arm)
#   ┌──────┐
#   │      │
#   │      ├──────────┐
#   │      │          │
#   └──────┴──────────┘
ZONE1_VERTICES = [
    (-300.0,  100.0),   # top-left of vertical arm
    (-150.0,  100.0),   # inner corner top
    (-150.0,  200.0),   # top of horizontal arm
    (  50.0,  200.0),   # top-right
    (  50.0,   50.0),   # bottom-right
    (-300.0,   50.0),   # bottom-left
]

# Zone 2 — irregular pentagon (public square)
ZONE2_VERTICES = [
    (-100.0, -200.0),
    ( 150.0, -220.0),
    ( 200.0,  -50.0),
    (  80.0,   30.0),
    (-120.0,  -20.0),
]

MAP_CONFIG = {
    'center':        [0.0, 0.0, 0.0],
    'size':          [1400, 1400],
    'cell_size':     (0.5, 0.5),
    'ground_height': 0.0,
}

# Experiment
NOISE_POWER   = 1e-10
JITTER_SEED   = 42
JITTER_MAG    = 1e-4
NUM_ITERATIONS = 50
BASELINES     = ["grad_full_rejection", "grad_full_triangulated", "grad_proportional"]
METRICS       = ["rsrp_mean_dbm", "rsrp_p10_dbm", "sir_median_db", "sir_p10_db", "coverage_fraction"]
OUTPUT_PATH   = "../scripts/report/london_experiment_results.json"

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../src'))
import warnings; warnings.filterwarnings("ignore")
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
import mitsuba as mi

try:
    import sionna.rt
except ImportError:
    os.system("pip install sionna-rt")
    import sionna.rt

from sionna.rt import load_scene, AntennaArray
from sionna.rt.antenna_pattern import antenna_pattern_registry

from scene_parser import extract_building_info
from tx_placement import TxPlacement
from boresight_pathsolver import create_zone_mask
from angle_utils import compute_initial_angles_from_position, azimuth_elevation_to_yaw_pitch
from multi_tx_optimizer import TxConfig
from experiment_runner import (
    ExperimentConfig, run_experiment_suite,
    compare_all_results, plot_cdf, plot_loss_curves, plot_metric_bars,
)

scene = load_scene(SCENE_XML_PATH)
scene.frequency = CARRIER_HZ

single_el = np.array([[0.0, 0.0, 0.0]])
scene.tx_array = AntennaArray(
    antenna_pattern=antenna_pattern_registry.get("tr38901")(polarization="V"),
    normalized_positions=single_el.T,
)
scene.rx_array = AntennaArray(
    antenna_pattern=antenna_pattern_registry.get("iso")(polarization="V"),
    normalized_positions=single_el.T,
)
for rm in scene.radio_materials.values():
    rm.scattering_coefficient = 0.4

building_info = extract_building_info(SCENE_XML_PATH, verbose=False)
print(f"Scene loaded  |  {CARRIER_HZ/1e9:.1f} GHz  |  {len(building_info)} buildings")

In [ ]:
# ── ZONE MASKS ───────────────────────────────────────────────────────────────
zone_params1 = {'vertices': ZONE1_VERTICES}
zone_mask1, look_at1, zone_stats1 = create_zone_mask(
    map_config=MAP_CONFIG,
    zone_type='polygon',
    zone_params=zone_params1,
    target_height=1.5,
    scene_xml_path=SCENE_XML_PATH,
    exclude_buildings=True,
)

zone_params2 = {'vertices': ZONE2_VERTICES}
zone_mask2, look_at2, zone_stats2 = create_zone_mask(
    map_config=MAP_CONFIG,
    zone_type='polygon',
    zone_params=zone_params2,
    target_height=1.5,
    scene_xml_path=SCENE_XML_PATH,
    exclude_buildings=True,
)

zone_masks = {TX1_NAME: zone_mask1, TX2_NAME: zone_mask2}

print(f"Zone 1 (L-shape):   {zone_stats1['num_cells']} cells  centroid={zone_stats1['centroid_xy']}")
print(f"Zone 2 (pentagon):  {zone_stats2['num_cells']} cells  centroid={zone_stats2['centroid_xy']}")

In [ ]:
# ── TX PLACEMENTS ────────────────────────────────────────────────────────────
tx_placer1 = TxPlacement(scene, TX1_NAME, SCENE_XML_PATH, TX1_BUILDING_ID, offset=TX1_HEIGHT_M)
tx_placer1.set_rooftop_zone_facing(zone_stats1["centroid_xy"])
tx1 = scene.get(TX1_NAME)
tx1_pos = tx1.position.numpy().flatten().tolist()
print(f"TX1 at ({tx1_pos[0]:.2f}, {tx1_pos[1]:.2f}, {tx1_pos[2]:.2f})")

tx_placer2 = TxPlacement(scene, TX2_NAME, SCENE_XML_PATH, TX2_BUILDING_ID, offset=TX2_HEIGHT_M)
tx_placer2.set_rooftop_zone_facing(zone_stats2["centroid_xy"])
tx2 = scene.get(TX2_NAME)
tx2_pos = tx2.position.numpy().flatten().tolist()
print(f"TX2 at ({tx2_pos[0]:.2f}, {tx2_pos[1]:.2f}, {tx2_pos[2]:.2f})")

In [ ]:
# ── INITIAL JITTER ───────────────────────────────────────────────────────────
rng = np.random.default_rng(JITTER_SEED)

base_az1, base_el1 = compute_initial_angles_from_position(tx1_pos, zone_stats1["look_at_xyz"])
initial_az1 = base_az1 + float(rng.uniform(-JITTER_MAG, JITTER_MAG))
initial_el1 = base_el1 + float(rng.uniform(-JITTER_MAG, JITTER_MAG))
yaw1, pitch1 = azimuth_elevation_to_yaw_pitch(initial_az1, initial_el1)
tx1.orientation = mi.Point3f(yaw1, pitch1, 0.0)

base_az2, base_el2 = compute_initial_angles_from_position(tx2_pos, zone_stats2["look_at_xyz"])
initial_az2 = base_az2 + float(rng.uniform(-JITTER_MAG, JITTER_MAG))
initial_el2 = base_el2 + float(rng.uniform(-JITTER_MAG, JITTER_MAG))
yaw2, pitch2 = azimuth_elevation_to_yaw_pitch(initial_az2, initial_el2)
tx2.orientation = mi.Point3f(yaw2, pitch2, 0.0)

# Store for restoration
_init = {
    TX1_NAME: (tx1_pos[:], yaw1, pitch1),
    TX2_NAME: (tx2_pos[:], yaw2, pitch2),
}

print(f"TX1  Az={initial_az1:.6f}°  El={initial_el1:.6f}°")
print(f"TX2  Az={initial_az2:.6f}°  El={initial_el2:.6f}°")

In [ ]:
# ── ZONE VISUALIZATION ───────────────────────────────────────────────────────
_cx, _cy, _ = MAP_CONFIG['center']
_w, _h = MAP_CONFIG['size']
extent = [_cx - _w/2, _cx + _w/2, _cy - _h/2, _cy + _h/2]

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(np.ma.masked_where(zone_mask1 == 0, zone_mask1),
          origin='lower', extent=extent, cmap='Greens', vmin=0, vmax=1, alpha=0.45)
ax.imshow(np.ma.masked_where(zone_mask2 == 0, zone_mask2),
          origin='lower', extent=extent, cmap='Blues',  vmin=0, vmax=1, alpha=0.45)

for bdata in building_info.values():
    verts = bdata['vertices'][:, :2]
    ax.add_patch(MplPolygon(verts, closed=True, facecolor='gray',
                            edgecolor='black', linewidth=0.8, alpha=0.4))

ax.plot(tx1_pos[0], tx1_pos[1], '^', color='darkgreen', markersize=14,
        markeredgecolor='black', label=f'TX1: {TX1_NAME} (bldg {TX1_BUILDING_ID})', zorder=5)
ax.plot(tx2_pos[0], tx2_pos[1], 's', color='steelblue', markersize=12,
        markeredgecolor='black', label=f'TX2: {TX2_NAME} (bldg {TX2_BUILDING_ID})', zorder=5)
ax.plot(*zone_stats1['centroid_xy'], 'o', color='darkgreen', markersize=8,
        markeredgecolor='k', label='Zone 1 centroid', zorder=5)
ax.plot(*zone_stats2['centroid_xy'], 'o', color='steelblue', markersize=8,
        markeredgecolor='k', label='Zone 2 centroid', zorder=5)

ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_title('London — Zone 1: L-shape (green)  |  Zone 2: Pentagon (blue)')
ax.legend(loc='upper right'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── CONVERGENCE HELPER ───────────────────────────────────────────────────────
def check_convergence(suite_output, baseline_id, tx_name, tol=1e-4):
    entry = suite_output["results"].get(baseline_id, {})
    opt   = entry.get("optimizer_result")
    if not isinstance(opt, dict):
        return {"status": "error"}
    tx_res  = opt.get(tx_name, {})
    az_hist = tx_res.get("az_history", [])
    el_hist = tx_res.get("el_history", [])
    if len(az_hist) > 1:
        for i in range(1, len(az_hist)):
            if abs(az_hist[i] - az_hist[i-1]) < tol and abs(el_hist[i] - el_hist[i-1]) < tol:
                return {"status": "converged", "iteration": i}
        return {"status": "no convergence", "iterations": len(az_hist)}
    init_angles = suite_output["metadata"]["initial_angles"].get(tx_name, [None, None])
    best_angles = tx_res.get("best_angles", [None, None])
    if None in list(init_angles) + list(best_angles):
        return {"status": "error"}
    d_az = abs(best_angles[0] - init_angles[0])
    d_el = abs(best_angles[1] - init_angles[1])
    label = "no movement" if d_az < tol and d_el < tol else "displaced"
    return {"status": label, "Δaz_deg": round(d_az, 4), "Δel_deg": round(d_el, 4)}

In [ ]:
# ── RUN SUITE ────────────────────────────────────────────────────────────────
tx_configs = [
    TxConfig(
        name=TX1_NAME,
        building_id=TX1_BUILDING_ID,
        zone_params=zone_params1,
        tx_height_offset=TX1_HEIGHT_M,
        num_sample_points=best_n,
    ),
    TxConfig(
        name=TX2_NAME,
        building_id=TX2_BUILDING_ID,
        zone_params=zone_params2,
        tx_height_offset=TX2_HEIGHT_M,
        num_sample_points=best_n,
    ),
]
exp_config = ExperimentConfig(
    baselines=BASELINES,
    noise_power=NOISE_POWER,
    num_iterations=NUM_ITERATIONS,
    output_path=OUTPUT_PATH,
)
suite_output = run_experiment_suite(
    scene=scene,
    tx_configs=tx_configs,
    map_config=MAP_CONFIG,
    scene_xml_path=SCENE_XML_PATH,
    zone_masks=zone_masks,
    exp_config=exp_config,
)

In [ ]:
# ── CONVERGENCE + RESULTS (RSRP & SINR only) ─────────────────────────────────
print(f"\n{'─'*60}")
print(f"  Convergence Analysis  (tol = {JITTER_MAG:.0e} deg)")
print(f"{'─'*60}")
for bid in BASELINES:
    print(f"  {bid}")
    for tx_name in [TX1_NAME, TX2_NAME]:
        conv = check_convergence(suite_output, bid, tx_name, tol=JITTER_MAG)
        print(f"    {tx_name:<8s}  {conv}")

print("\n")
df = compare_all_results(suite_output, metrics=METRICS)

In [ ]:
# ── VISUALIZATION ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

plot_metric_bars(
    suite_output,
    metrics=("rsrp_mean_dbm", "rsrp_p10_dbm", "sir_median_db", "sir_p10_db"),
    ax=axes[0],
)
axes[0].set_title("RSRP & SINR — All TXs (avg)")

plot_loss_curves(suite_output, ax=axes[1])
axes[1].set_title("Convergence Curves")

plot_cdf(suite_output, metric="rsrp_values_dbm", ax=axes[2])
axes[2].set_title("RSRP CDF")

plt.suptitle(f"London Experiment — 2 TX, Organic Zones, 3.5 GHz (n={best_n})", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── PER-TX BREAKDOWN ─────────────────────────────────────────────────────────
from experiment_runner import get_distribution_data

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for row_i, tx_name in enumerate([TX1_NAME, TX2_NAME]):
    for col_i, metric in enumerate(["rsrp_values_dbm", "sir_values_db"]):
        ax = axes[row_i][col_i]
        plot_cdf(suite_output, metric=metric, tx_name=tx_name, ax=ax)
        ax.set_title(f"{tx_name} — {metric.replace('_values_', ' ').replace('_dbm','(dBm)').replace('_db','(dB)')}")

plt.suptitle("Per-TX CDF: RSRP and SINR", y=1.02)
plt.tight_layout()
plt.show()